# S3 Data Quality Notebook

Ноутбук для просмотра содержимого S3, загрузки parquet-данных в единый `DataFrame` и быстрой проверки качества данных.

In [ ]:
import io
import os
from typing import Iterable, Optional

import boto3
import pandas as pd
import yaml
from dotenv import load_dotenv

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

load_dotenv()

In [ ]:
with open("config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

DEFAULT_BUCKET = config["storage"]["bucket"]
DEFAULT_PREFIX = config["storage"]["prefix"]
DEFAULT_SOURCE = config["source"]
DEFAULT_SYMBOL = config["symbols"][0]

DEFAULT_BUCKET, DEFAULT_PREFIX, DEFAULT_SOURCE, DEFAULT_SYMBOL

In [ ]:
def make_s3_client():
    return boto3.client(
        "s3",
        endpoint_url=os.getenv("YC_ENDPOINT"),
        region_name=os.getenv("YC_REGION"),
        aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
        aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
    )


s3 = make_s3_client()

In [ ]:
def build_partition_prefix(
    base_prefix: str,
    source: str,
    symbol: Optional[str] = None,
    interval: Optional[str] = None,
    date_from: Optional[str] = None,
    date_to: Optional[str] = None,
) -> str:
    parts = [base_prefix.strip("/"), source]

    if symbol:
        parts.append(f"symbol={symbol}")

    if interval:
        parts.append(f"interval={interval}")

    prefix = "/".join(part for part in parts if part)

    if date_from and date_to and date_from == date_to:
        prefix = f"{prefix}/date={date_from}"

    return prefix.rstrip("/") + "/"


def list_s3_objects(bucket: str, prefix: str, max_keys: Optional[int] = None) -> pd.DataFrame:
    paginator = s3.get_paginator("list_objects_v2")
    rows = []
    total = 0

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            rows.append(
                {
                    "key": obj["Key"],
                    "size_bytes": obj["Size"],
                    "last_modified": obj["LastModified"],
                }
            )
            total += 1
            if max_keys is not None and total >= max_keys:
                return pd.DataFrame(rows)

    return pd.DataFrame(rows)


def _date_from_key(key: str) -> Optional[str]:
    for part in key.split("/"):
        if part.startswith("date="):
            return part.split("=", 1)[1]
    return None


def _read_parquet_from_s3(bucket: str, key: str) -> pd.DataFrame:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    return pd.read_parquet(io.BytesIO(body))


def build_df(
    bucket: str,
    base_prefix: str,
    source: str,
    symbol: Optional[str] = None,
    interval: Optional[str] = None,
    date_from: Optional[str] = None,
    date_to: Optional[str] = None,
    limit_files: Optional[int] = None,
) -> pd.DataFrame:
    """Собирает единый DataFrame из parquet-файлов в S3.

    Параметры date_from/date_to работают как фильтр по партициям вида date=YYYY-MM-DD.
    """
    prefix = build_partition_prefix(
        base_prefix=base_prefix,
        source=source,
        symbol=symbol,
        interval=interval,
        date_from=date_from,
        date_to=date_to,
    )

    objects_df = list_s3_objects(bucket=bucket, prefix=prefix)
    if objects_df.empty:
        raise FileNotFoundError(f"No objects found for prefix: s3://{bucket}/{prefix}")

    objects_df = objects_df[objects_df["key"].str.endswith(".parquet")].copy()
    objects_df["partition_date"] = objects_df["key"].map(_date_from_key)

    if date_from:
        objects_df = objects_df[objects_df["partition_date"] >= date_from]

    if date_to:
        objects_df = objects_df[objects_df["partition_date"] <= date_to]

    objects_df = objects_df.sort_values(["partition_date", "key"]).reset_index(drop=True)

    if limit_files is not None:
        objects_df = objects_df.head(limit_files)


    frames = []
    for key in objects_df["key"]:
        df_part = _read_parquet_from_s3(bucket=bucket, key=key)
        df_part["source_file"] = key
        df_part["partition_date"] = _date_from_key(key)
        frames.append(df_part)

    df = pd.concat(frames, ignore_index=True)

    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
        df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)

    return df


In [ ]:
objects = list_s3_objects(
    bucket=DEFAULT_BUCKET,
    prefix=build_partition_prefix(
        base_prefix=DEFAULT_PREFIX,
        source=DEFAULT_SOURCE,
        symbol=DEFAULT_SYMBOL,
    ),
    max_keys=50,
)

objects.head(20)

In [ ]:
df = build_df(
    bucket=DEFAULT_BUCKET,
    base_prefix=DEFAULT_PREFIX,
    source=DEFAULT_SOURCE,
    symbol=DEFAULT_SYMBOL,
    date_from="2021-04-30",
    date_to="2021-05-03",
)

df.head()

In [ ]:
df.info()

In [ ]:
quality_report = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "nulls": df.isna().sum(),
        "null_pct": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(dropna=False),
    }
).sort_values(["null_pct", "nulls"], ascending=False)

quality_report

In [ ]:
if "timestamp" in df.columns:
    print("min timestamp:", df["timestamp"].min())
    print("max timestamp:", df["timestamp"].max())
    print("duplicates:", df["timestamp"].duplicated().sum())

df.describe(include="all").T